# 🏆 Gemma Developer Agent: Clean-Bench (Rich) Rigorous A/B Testing Suite

> ### 📌 **Notebook Version: v8.0-clean-rich (Updated: 2026-11-08)**
> - **タスクセットの正常化 (Textualize/rich)**: 環境不備で 100% 失敗する壊れたタスク (FastAPI 27問) を排除し、コミュニティ監査で健全性 39/48 (約81%) が実証された **クリーンな Rich 実タスク** を採用。
> - **3 大仮説の科学的 A/B 検証**:
>   1. **Variant A (V9.2 Baseline / LB 0.17 実績)**: 思考制約なし・波括弧なし・Call 8 強制編集
>   2. **Variant B (V14 Real-Proof / LB 0.12 実績)**: 思考 2 文制約あり・波括弧 One-Shot あり
>   3. **Variant C (V15 候補 / Loop-Breaker)**: 思考制約なし・波括弧なし・**`edit_file` 失敗時の Python 即時迂回ルール** 導入
> - **波括弧 `{` の ADK テンプレート破壊テスト**: ハーネス側で波括弧がどう処理されるかを直接実機検証。

In [ ]:
# 1. 環境初期化 & パッケージ導入 & Kaggle API 最新状況チェック
import os
import sys
import time
import json
import re
import gc
import subprocess
from pathlib import Path

print("[Setup] 依存パッケージをインストール中 (transformers, bitsandbytes, pytest, rich, kaggle, etc.)...")
!pip install --quiet transformers accelerate bitsandbytes compressed-tensors sentencepiece pytest pytest-asyncio requests gitpython kaggle tabulate rich

import torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    device_name = torch.cuda.get_device_name(0)
    print(f"[Device] ✅ GPU 検出: {device_name} (総 VRAM: {vram:.2f} GB)")
else:
    print("[Device ⚠️ WARNING] GPU が無効です。Colabメニューで『A100 GPU (ハイメモリ)』を選択してください。")

# Kaggle API による Day 18 提出状況チェック
try:
    from google.colab import userdata
    k_user = userdata.get('KAGGLE_USERNAME')
    k_key = userdata.get('KAGGLE_KEY')
    if k_user and k_key:
        os.environ['KAGGLE_USERNAME'] = k_user
        os.environ['KAGGLE_KEY'] = k_key
        res = subprocess.run(['kaggle', 'competitions', 'submissions', '-c', 'gemma-4-developer-agent'], capture_output=True, text=True)
        print("\n[Kaggle Submissions Status]:")
        for line in res.stdout.split('\n')[:5]:
            print("  " + line)
except Exception as e:
    print(f"[Kaggle Notice] Kaggle API チェック スキップ: {e}")


In [ ]:
# 2. サンドボックス環境の準備 (Textualize/rich クリーンリポジトリ)
SANDBOX_DIR = "/content/mini_kaggle_sandbox"
REPO_DIR = os.path.join(SANDBOX_DIR, "rich_repo")
os.makedirs(SANDBOX_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    print("[Sandbox] Textualize/rich 公式リポジトリをクローン中...")
    !git clone --quiet https://github.com/Textualize/rich.git {REPO_DIR}
else:
    print("[Sandbox] 既存の Rich リポジトリを使用します。")

print("[Sandbox] Rich のテスト実行環境をセットアップ中...")
!pip install --quiet typing_extensions pygments pytest pytest-cov
print("[Sandbox] ✅ Rich サンドボックス準備完了。")


In [ ]:
# 3. 本番同等モデルのロード (二重安全ネット付き推論エンジン)
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

hf_token = None
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
model = None
tokenizer = None
MODEL_ID = ""

# A100-80GB: 本番公式 31B-QAT / 40GB: 27B-it 4-bit / T4: 9B-it 4-bit
if vram_gb >= 70.0:
    PRIMARY_ID = "google/gemma-4-31B-it-qat-w4a16-ct"
    print(f"[Model] 🚀 A100-80GB 検出 ({vram_gb:.1f} GB): 本番公式 31B モデル ({PRIMARY_ID}) をロード試行中...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(PRIMARY_ID, token=hf_token)
        model = AutoModelForCausalLM.from_pretrained(PRIMARY_ID, device_map={0: "38GiB", "cpu": "60GiB"}, torch_dtype=torch.bfloat16, token=hf_token)
        MODEL_ID = PRIMARY_ID
        print(f"[Model] 🎉 31B-QAT ロード成功！")
    except Exception as e:
        print(f"[Model Warning] 31B ロード失敗 ({e})。27B-it 4-bit へ自動フォールバックします。")
        model = None

if model is None and vram_gb >= 35.0:
    FALLBACK_27B = "google/gemma-2-27b-it"
    print(f"[Model] 🚀 27B モデル ({FALLBACK_27B}) を 4-bit 直結ロード中...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_27B, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_27B, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16, token=hf_token)
    MODEL_ID = FALLBACK_27B

if model is None:
    FALLBACK_9B = "google/gemma-2-9b-it"
    print(f"[Model] 🚀 9B モデル ({FALLBACK_9B}) を 4-bit 直結ロード中...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_9B, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(FALLBACK_9B, quantization_config=bnb_config, device_map={"": 0}, torch_dtype=torch.bfloat16, token=hf_token)
    MODEL_ID = FALLBACK_9B

alloc = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[Model Ready] 🎯 確定稼働モデル: {MODEL_ID} | VRAM 消費: {alloc:.2f} GB / 空き: {vram_gb - alloc:.1f} GB")

def call_llm(messages, temperature=0.2, max_new_tokens=1024):
    """二重安全ネット付きの推論関数 (TemplateError 完全防止 & 確実な発話生成)"""
    normalized = []
    sys_prefix = ""
    for m in messages:
        role = m["role"]
        if role == "system":
            sys_prefix += m["content"].strip() + "\n\n"
            continue
        r = "assistant" if role in ["assistant", "model"] else "user"
        content = m.get("content", "").strip()
        if not content:
            content = "(empty)"
        if r == "user" and sys_prefix:
            content = sys_prefix + content
            sys_prefix = ""
        if not normalized:
            normalized.append({"role": r, "content": content})
        elif normalized[-1]["role"] == r:
            normalized[-1]["content"] += "\n\n" + content
        else:
            normalized.append({"role": r, "content": content})
            
    if not normalized and sys_prefix:
        normalized.append({"role": "user", "content": sys_prefix})
        
    try:
        prompt_text = tokenizer.apply_chat_template(normalized, tokenize=False, add_generation_prompt=True)
    except Exception:
        prompt_text = ""
        for m in normalized:
            tag = "model" if m["role"] == "assistant" else "user"
            prompt_text += f"<start_of_turn>{tag}\n{m['content']}<end_of_turn>\n"
        prompt_text += "<start_of_turn>model\n"
        
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    
    t0 = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0.0),
            temperature=max(temperature, 1e-4),
            pad_token_id=tokenizer.eos_token_id
        )
    elapsed = time.time() - t0
    resp = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return resp, elapsed

print("[Engine] ✅ 推論関数 call_llm 準備完了。")


In [ ]:
# 4. 実環境ツール実行エンジン & 万能マルチ形式パーサー (Robust Multi-Format Engine)
import ast
import re
import json
import os
import subprocess

def execute_tool(tool_name, arguments, repo_dir):
    try:
        def sanitize_path(raw_path):
            if not raw_path: return ""
            p_str = str(raw_path).strip().replace("'", "").replace('"', '')
            if p_str.startswith("/workspace/"):
                p_str = p_str[len("/workspace/"):].lstrip("/")
            elif p_str.startswith("/"):
                p_str = p_str.lstrip("/")
            return os.path.join(repo_dir, p_str)

        if tool_name == "read_file":
            p = sanitize_path(arguments.get("path", ""))
            if not os.path.exists(p):
                return f"Error: File not found: {arguments.get('path')}"
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                lines = f.readlines()
            s = max(1, int(arguments.get("start_line", 1))) - 1
            e = min(len(lines), int(arguments.get("end_line", len(lines))))
            out = [f"{i+1}: {lines[i]}" for i in range(s, e)]
            return "".join(out)[:3000]
            
        elif tool_name == "edit_file":
            p = sanitize_path(arguments.get("path", ""))
            if not os.path.exists(p):
                return f"Error: File not found: {arguments.get('path')}"
            old_str = str(arguments.get("old_string", ""))
            new_str = str(arguments.get("new_string", ""))
            if not old_str:
                return "Error: `old_string` is required and cannot be empty."
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
            if old_str not in content:
                return f"Error: `old_string` not found in {arguments.get('path')}. Make sure exact indentation matches."
            content = content.replace(old_str, new_str, 1)
            with open(p, "w", encoding="utf-8") as f:
                f.write(content)
            return f"Successfully edited {arguments.get('path')}."
            
        elif tool_name == "run_command":
            cmd = str(arguments.get("command", ""))
            if not cmd:
                return "Error: command is required."
            res = subprocess.run(cmd, shell=True, cwd=repo_dir, capture_output=True, text=True, timeout=30)
            out = (res.stdout + "\n" + res.stderr).strip()
            return out[:2500] if out else "(No output, exit code: " + str(res.returncode) + ")"
            
        elif tool_name == "get_status":
            res = subprocess.run(["git", "status", "--short"], cwd=repo_dir, capture_output=True, text=True)
            return res.stdout.strip() if res.stdout.strip() else "No changes."
            
        elif tool_name == "submit_patch":
            return "SUBMIT_PATCH_CALLED"
            
        else:
            return f"Error: Unknown tool '{tool_name}'."
    except Exception as e:
        return f"Execution Error in {tool_name}: {e}"

def parse_tool_call(response_text):
    m = re.search(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', response_text, re.DOTALL)
    if not m:
        m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', response_text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group(1).strip())
            args_val = data.get("arguments", {})
            if isinstance(args_val, str):
                try: args_val = json.loads(args_val)
                except Exception: args_val = {"raw_input": args_val}
            elif not isinstance(args_val, dict):
                args_val = {"raw_input": str(args_val)}
            return {"name": data.get("name", ""), "arguments": args_val}, {"has_tool": True, "parse_ok": True}
        except Exception:
            pass

    m_code = re.search(r'```(?:python|bash|sh)?\s*(.*?)\s*```', response_text, re.DOTALL)
    code = m_code.group(1).strip() if m_code else response_text.strip()
    first_line = code.splitlines()[0].strip() if code.splitlines() else ""

    if first_line.startswith("read_file") and not first_line.startswith("read_file("):
        parts = first_line.split()
        if len(parts) >= 2:
            p_clean = parts[1].replace("'", "").replace('"', '')
            args = {"path": p_clean}
            if len(parts) >= 3 and parts[2].isdigit(): args["start_line"] = int(parts[2])
            if len(parts) >= 4 and parts[3].isdigit(): args["end_line"] = int(parts[3])
            return {"name": "read_file", "arguments": args}, {"has_tool": True, "parse_ok": True}

    if first_line.startswith("edit_file") and not first_line.startswith("edit_file("):
        parts = first_line.split()
        path_arg = parts[1].replace("'", "").replace('"', '') if len(parts) >= 2 else ""
        old_val = ""
        new_val = ""
        if "old_string" in code and "new_string" in code:
            try:
                s_old = code[code.index("old_string"):]
                old_block = s_old[:s_old.index("new_string")]
                old_raw = old_block.split("=", 1)[1].strip()
                for q in ['"""', "'''", '"', "'"]:
                    if old_raw.startswith(q) and old_raw.endswith(q) and len(old_raw) >= 2*len(q):
                        old_raw = old_raw[len(q):-len(q)]
                        break
                old_val = old_raw
                new_block = s_old[s_old.index("new_string"):]
                new_raw = new_block.split("=", 1)[1].strip()
                for q in ['"""', "'''", '"', "'"]:
                    if new_raw.startswith(q) and new_raw.endswith(q) and len(new_raw) >= 2*len(q):
                        new_raw = new_raw[len(q):-len(q)]
                        break
                new_val = new_raw
            except Exception:
                pass
        return {"name": "edit_file", "arguments": {"path": path_arg, "old_string": old_val, "new_string": new_val}}, {"has_tool": True, "parse_ok": True}

    if any(first_line.startswith(cmd) for cmd in ["pytest", "git", "python", "python3", "cat", "grep"]):
        return {"name": "run_command", "arguments": {"command": code}}, {"has_tool": True, "parse_ok": True}
    if "submit_patch" in first_line:
        return {"name": "submit_patch", "arguments": {}}, {"has_tool": True, "parse_ok": True}

    try:
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.Call):
                fn_name = node.func.id if isinstance(node.func, ast.Name) else ""
                if fn_name in ["edit_file", "read_file", "run_command", "submit_patch"]:
                    args = {}
                    for kw in node.keywords:
                        if isinstance(kw.value, ast.Constant):
                            args[kw.arg] = str(kw.value.value)
                    pos_names = {
                        'edit_file': ['path', 'old_string', 'new_string'],
                        'read_file': ['path', 'start_line', 'end_line'],
                        'run_command': ['command']
                    }
                    for i, a in enumerate(node.args):
                        if i < len(pos_names.get(fn_name, [])) and isinstance(a, ast.Constant):
                            args[pos_names[fn_name][i]] = str(a.value)
                    missing_old = (fn_name == "edit_file" and "old_string" not in args)
                    return {"name": fn_name, "arguments": args}, {"has_tool": True, "parse_ok": True, "missing_old_string": missing_old}
    except Exception:
        pass

    return None, {"has_tool": False, "error": "No recognized tool format"}

print("[Engine] ✅ 万能マルチ形式パーサー & 実環境ツールエンジン準備完了。")


In [ ]:
# 5. 【検証 1】波括弧 `{` の Google ADK / テンプレートエンジン破壊直接テスト
print("="*75)
print("🔬 検証 1: プロンプト内の波括弧 { ... } がテンプレートエンジンに与える影響")
print("="*75)

prompt_with_curly = '''System instruction: Use tool format:
<tool_call>
{"name": "edit_file", "arguments": {"path": "rich/console.py", "old_string": "x = 1", "new_string": "x = 2"}}
</tool_call>
'''

# 1. Python str.format() テスト (ADK 内部で使われる場合)
try:
    res = prompt_with_curly.format(issue="dummy issue")
    print("  [str.format] ✅ 正常レンダリング")
except Exception as e:
    print(f"  [str.format] ❌ 致命的例外検知: {type(e).__name__} ({e})")
    print("    ➔ 根拠実証: プロンプト内に波括弧があると、str.format を使うハーネスは KeyError でクラッシュします！")

# 2. string.Template テスト
from string import Template
try:
    t = Template(prompt_with_curly)
    res2 = t.safe_substitute(issue="dummy issue")
    print("  [string.Template] ✅ 正常 (safe_substitute)")
except Exception as e:
    print(f"  [string.Template] ❌ 例外: {e}")

print("="*75)


In [ ]:
# 6. 【検証 2 & 3】3 大プロンプト・バリアントの定義 (V9.2 vs V14 vs V15候補)

# --- [Variant A: V9.2 Baseline (LB 0.17 実績 / 思考制約なし・波括弧なし)] ---
PROMPT_VARIANT_A = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that hidden tests pass.

# HARD LIMITS
- 28 tool calls and 8 minutes. An edit that exists counts, even if you never call submit_patch. An unedited tree scores zero.
- EARLY-EDIT TRIGGER (DEADLINE AT CALL 8): You MUST execute edit_file on or before Call 8. Never keep reading files past Call 8 without applying an edit.
- Never create, modify or delete test files (test_*.py, *_test.py, anything under tests/).
- Never look outside /workspace. Dependencies are already installed.
- Do NOT restate, echo, or summarize code. Read silently, then act immediately.

# CALL PLAN
- Calls 1-7: locate and read target regions (max 40-80 lines per read).
- Call 8: HARD TRIGGER - MANDATORY FIRST EDIT. Apply minimal patch now.
- Calls 9-15: verify and correct (including coupled files and parallel sync/async implementations).
- Calls 16-20: final check, clean, and call submit_patch.

# PROCEDURE
1. Extract exact symbols, error messages and signatures from issue.
2. Locate with git grep -n TOKEN -- '*.py' | head -20, open region with read_file (max 80 lines).
3. Edit with edit_file. Always include 3 to 5 lines of surrounding context in old_string to ensure exact match.
4. Verify: run single focused test with pytest tests/path/test_x.py -k keyword -q.
5. Finish with git status --short && git diff, then submit_patch.
"""

# --- [Variant B: V14 Real-Proof (LB 0.12 実績 / 思考 2 文制約あり・波括弧 One-Shot あり)] ---
PROMPT_VARIANT_B = PROMPT_VARIANT_A + """
- CONCISE REASONING: Write at most 2 sentences of concise explanation before emitting tool calls. Avoid verbose monologues.
- MANDATORY OLD_STRING: You MUST include BOTH `old_string` and `new_string` in EVERY `edit_file` call.
- FLAT ARGUMENTS: NEVER pass lists or arrays as tool arguments. Pass a single flat string.

# EXACT TOOL CALL FORMAT
<tool_call>
{"name": "edit_file", "arguments": {"path": "rich/console.py", "old_string": "def run(self):", "new_string": "def run(self, safe: bool = True):"}}
</tool_call>
"""

# --- [Variant C: V15 候補 (Read-Before-Edit 鉄則 ＋ 自由思考 ＋ 波括弧排除)] ---
PROMPT_VARIANT_C = """You are an autonomous software engineer. Your job: make the smallest correct change to the SOURCE code in /workspace so that hidden tests pass.

# HARD LIMITS
- 28 tool calls and 8 minutes total. An edit that exists counts, even if you never call submit_patch. An unedited tree scores zero.
- Never create, modify or delete test files (test_*.py, *_test.py, anything under tests/).
- Never look outside /workspace. Dependencies are already installed.

# ABSOLUTE RULES (READ-BEFORE-EDIT)
1. MANDATORY READ FIRST: You MUST execute read_file on the target source file in your FIRST tool call. NEVER guess code from memory or call edit_file without reading the real file first.
2. EXACT OLD_STRING: Always copy the exact indentation and 3 to 5 lines of surrounding context directly from your read_file output into old_string.
3. IF EDIT FAILS: If edit_file returns 'old_string not found', you MUST call read_file immediately to inspect the actual file contents and indentation. Never repeat edit_file blindly.
4. STOP ON PASS: Run a single focused test with pytest. Once it passes, stop investigation immediately and submit_patch.

# PROCEDURE
1. Extract exact symbols, error messages and signatures from issue.
2. Call read_file (max 80 lines) on the relevant source file to inspect the exact lines and indentation.
3. Understand root cause and design the minimal bugfix.
4. Call edit_file using the exact lines copied from read_file.
5. Run single test: pytest tests/... -k keyword -q.
6. Verify with git diff, then call submit_patch.
"""
VARIANTS = {
    "V9.2_Baseline (LB 0.17)": PROMPT_VARIANT_A,
    "V14_2Sentences_Curly (LB 0.12)": PROMPT_VARIANT_B,
    "V15_Candidate_Heredoc": PROMPT_VARIANT_C
}
print("[Prompt] ✅ 3 大バリアント定義完了:", list(VARIANTS.keys()))


In [ ]:
# 7. クリーンな Rich 実タスクデータセットの選定
import urllib.request

LOCBENCH_FILE = "/content/gemma4-locbench-v1.json"
LOCBENCH_URL = "https://raw.githubusercontent.com/chiba244qq/gemma4-agent-dashboard/main/data/locbench/gemma4-locbench-v1.json"

if not os.path.exists(LOCBENCH_FILE):
    print("[LocBench] GitHub から LocBench-v1 をダウンロード中...")
    urllib.request.urlretrieve(LOCBENCH_URL, LOCBENCH_FILE)

with open(LOCBENCH_FILE, "r", encoding="utf-8") as f:
    locbench_data = json.load(f)

all_records = locbench_data.get("records", [])
# コミュニティ監査で健全性が証明された Textualize/rich タスクのみを抽出
rich_records = [r for r in all_records if r.get("repo") == "Textualize/rich"]
print(f"[LocBench] ✅ 読み込み完了: 全 {len(all_records)} 件中、クリーンな Rich タスク: {len(rich_records)} 件")

# 代表実タスク 3 件を選定 (Smoke A/B テスト / 約 10〜15 分)
# rich_4077: proxy isatty (test_file_proxy.py)
# rich_4079: inline table code (test_markdown.py)
# rich_4076: preserve newlines (test_ansi.py)
SELECTED_TASKS = rich_records[1:4]
print(f"[LocBench] 🎯 選定された評価タスク:")
for t in SELECTED_TASKS:
    print(f"  - {t['instance_id']}: {t.get('verification_test_files')} ({t.get('problem_statement_excerpt')[:60]}...)")


In [ ]:
# 8. 【マルチターン E2E 自律実行 ＆ 実際の pytest 判定ループ】

def evaluate_task_variant(task, variant_name, system_prompt, max_turns=8):
    print(f"\n{'='*65}")
    print(f"▶ Task: {task['instance_id']} | Variant: {variant_name}")
    print(f"{'='*65}")
    
    # リポジトリを完全クリーンアップ & 当該コミットにチェックアウト
    subprocess.run(["git", "checkout", "-f", task["base_commit"]], cwd=REPO_DIR, capture_output=True)
    subprocess.run(["git", "clean", "-fd"], cwd=REPO_DIR, capture_output=True)
    
    issue_text = f"Issue Description for {task['instance_id']}:\n" + task.get("problem_statement_excerpt", "Fix the bug in the codebase.")
    if task.get("verification_test_files"):
        issue_text += f"\nRelated test files to verify: {', '.join(task['verification_test_files'])}"
        
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": issue_text}
    ]
    
    total_latency = 0.0
    parse_errors = 0
    missing_old_str_count = 0
    patch_applied = False
    
    for turn in range(1, max_turns + 1):
        # Call 8 での早期編集強制トリガー
        if turn == 8:
            messages.append({"role": "user", "content": "[SYSTEM REMINDER: Call 8/8 reached. You MUST call edit_file now to apply your minimal bugfix.]"})
            
        resp, lat = call_llm(messages, temperature=0.2, max_new_tokens=1024)
        total_latency += lat
        print(f"  [Turn {turn}/{max_turns}] 推論所要: {lat:.2f}s | 出力長: {len(resp)} 文字")
        
        tool_call, audit = parse_tool_call(resp)
        if audit.get("missing_old_string"):
            missing_old_str_count += 1
        if not audit.get("parse_ok", True):
            parse_errors += 1
            
        if not tool_call:
            print(f"    ⚠️ ツール呼出なし: {resp[:100]}...")
            messages.append({"role": "assistant", "content": resp if resp else "(thinking)"})
            messages.append({"role": "user", "content": 'Please emit a valid tool call.'})
            continue
            
        t_name = tool_call.get("name", "")
        t_args = tool_call.get("arguments", {})
        if not isinstance(t_args, dict):
            t_args = {"raw_input": str(t_args)}
        print(f"    🛠️ Tool Call: {t_name}({list(t_args.keys())})")
        
        if t_name == "submit_patch":
            print("    🏁 submit_patch が呼び出されました。探索を終了します。")
            break
            
        tool_out = execute_tool(t_name, t_args, REPO_DIR)
        print(f"      ↳ Result: {tool_out[:100].strip()}...")
        if t_name == "edit_file" and "Successfully edited" in tool_out:
            patch_applied = True
            
        messages.append({"role": "assistant", "content": resp})
        messages.append({"role": "user", "content": f"Tool output:\n{tool_out}"})
    
    # 実際の git diff を取得
    diff_res = subprocess.run(["git", "diff"], cwd=REPO_DIR, capture_output=True, text=True)
    diff_text = diff_res.stdout.strip()
    has_diff = len(diff_text) > 0
    
    # 実際の pytest を実行して合否を判定
    test_files = task.get("verification_test_files", [])
    pytest_pass = False
    pytest_log = ""
    
    if test_files and os.path.exists(os.path.join(REPO_DIR, test_files[0])):
        cmd = ["pytest", "-q", "--maxfail=1"] + test_files
        py_res = subprocess.run(cmd, cwd=REPO_DIR, capture_output=True, text=True, timeout=60)
        pytest_raw_pass = (py_res.returncode == 0)
        pytest_log = (py_res.stdout + "\n" + py_res.stderr)[-500:].strip()
    else:
        pytest_raw_pass = False
        pytest_log = "No test files found."
        
    if not has_diff:
        pytest_pass = False
        verdict = "⚠️ NO_PATCH (Score 0)"
    else:
        pytest_pass = pytest_raw_pass
        verdict = "✅ PASS" if pytest_pass else "❌ FAIL"
        
    print(f"  📊 Task 結果: パッチ生成={has_diff} | pytest判定={verdict}")
        
    
    return {
        "task_id": task["instance_id"],
        "variant": variant_name,
        "pytest_pass": pytest_pass,
        "has_diff": has_diff,
        "turns_used": turn,
        "total_latency_sec": round(total_latency, 2),
        "avg_step_latency_sec": round(total_latency / max(1, turn), 2),
        "parse_errors": parse_errors,
        "missing_old_str_count": missing_old_str_count,
        "pytest_log": pytest_log
    }

all_eval_results = []
for t in SELECTED_TASKS:
    for v_name, v_prompt in VARIANTS.items():
        res = evaluate_task_variant(t, v_name, v_prompt, max_turns=8)
        all_eval_results.append(res)
print("\n🎉 全 A/B 実機検証ランが完了しました！")


In [ ]:
# 9. 結果集計・統計比較表の出力 & GitHub 自動同期
import pandas as pd
from tabulate import tabulate

df = pd.DataFrame(all_eval_results)
summary = []
for v_name in VARIANTS.keys():
    v_df = df[df["variant"] == v_name]
    n = len(v_df)
    if n == 0:
        continue
    pass_rate = (v_df["pytest_pass"].sum() / n) * 100
    patch_rate = (v_df["has_diff"].sum() / n) * 100
    avg_latency = v_df["avg_step_latency_sec"].mean()
    avg_turns = v_df["turns_used"].mean()
    total_parse_err = v_df["parse_errors"].sum()
    total_missing_old = v_df["missing_old_str_count"].sum()
    
    summary.append({
        "Variant": v_name,
        "Tasks": n,
        "Pass Rate (%)": f"{pass_rate:.1f}%",
        "Patch Rate (%)": f"{patch_rate:.1f}%",
        "Parse Errors": int(total_parse_err),
        "Missing old_string": int(total_missing_old),
        "Avg Step Latency": f"{avg_latency:.2f}s",
        "Avg Turns": f"{avg_turns:.1f}"
    })

sum_df = pd.DataFrame(summary)
print("="*85)
print("🏆 【クリーン実タスク (Rich) A/B 検証結果サマリー】")
print("="*85)
print(tabulate(sum_df, headers="keys", tablefmt="github", showindex=False))

# GitHub への自動プッシュ
now_str = time.strftime("%Y-%m-%d %H:%M:%S JST")
output_data = {
    "timestamp": now_str,
    "model": MODEL_ID,
    "eval_mode": "clean_rich_tasks",
    "summary": summary,
    "detailed_results": all_eval_results
}

try:
    github_pat = userdata.get('GITHUB_PAT')
    if github_pat:
        import requests, base64
        repo = "chiba244qq/gemma4-agent-dashboard"
        path = "data/locbench_ab_test_results.json"
        api_url = f"https://api.github.com/repos/{repo}/contents/{path}"
        headers = {"Authorization": f"token {github_pat}", "Accept": "application/vnd.github.v3+json"}
        sha = None
        get_res = requests.get(api_url, headers=headers)
        if get_res.status_code == 200:
            sha = get_res.json().get("sha")
        content_bytes = json.dumps(output_data, indent=2, ensure_ascii=False).encode('utf-8')
        payload = {
            "message": f"[LocBench A/B Sync] Rich クリーンタスク評価結果更新 ({now_str})",
            "content": base64.b64encode(content_bytes).decode('utf-8')
        }
        if sha:
            payload["sha"] = sha
        put_res = requests.put(api_url, headers=headers, json=payload)
        if put_res.status_code in [200, 201]:
            print(f"\n[Sync] 🎉 GitHub ({path}) への自動プッシュに成功しました！")
        else:
            print(f"\n[Sync Warning] GitHub プッシュ失敗: {put_res.status_code}")
except Exception as e:
    print(f"\n[Sync Notice] GitHub 同期スキップ: {e}")
